# Crop EDA – Cucumber & Sunflower Datasets


In [16]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from PIL import Image, ImageStat

CUCUMBER_ROOT = Path('archive')
SUNFLOWER_ROOT = Path('Sunflower Compressed')
IMGS_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def all_images(folder):
    """Recursively collect all image files under a folder."""
    return [p for p in folder.rglob('*') if p.suffix.lower() in IMGS_EXTS]

def class_images(root):
    """Return dict of class_name -> list of image paths (recursive per class subfolder)."""
    result = {}
    for d in sorted(root.iterdir()):
        if d.is_dir():
            imgs = all_images(d)
            if imgs:
                result[d.name] = imgs
    return result

cucumber_cls = class_images(CUCUMBER_ROOT)
sunflower_cls = class_images(SUNFLOWER_ROOT)
print('Cucumber classes:', list(cucumber_cls.keys()))
print('Sunflower classes:', list(sunflower_cls.keys()))


Cucumber classes: ['Flowering', 'Fruiting - 1', 'Fruiting - 2', 'Healthy Leaves', 'Unhealthy Leaves']
Sunflower classes: ['EarlyBloom', 'Healthy', 'MatureBud', 'Wilted', 'YoungBud']


## Image counts per class


In [17]:
c_counts = {k: len(v) for k, v in cucumber_cls.items()}
s_counts = {k: len(v) for k, v in sunflower_cls.items()}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(c_counts.keys(), c_counts.values(), color='steelblue')
axes[0].set_title('Cucumber – images per class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(s_counts.keys(), s_counts.values(), color='goldenrod')
axes[1].set_title('Sunflower – images per class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('counts.png', dpi=100, bbox_inches='tight')
plt.show()
print('Cucumber counts:', c_counts)
print('Sunflower counts:', s_counts)


Cucumber counts: {'Flowering': 362, 'Fruiting - 1': 428, 'Fruiting - 2': 442, 'Healthy Leaves': 41, 'Unhealthy Leaves': 132}
Sunflower counts: {'EarlyBloom': 1078, 'Healthy': 1202, 'MatureBud': 1008, 'Wilted': 1037, 'YoungBud': 1025}


C:\Users\akank\AppData\Local\Temp\ipykernel_19272\2868949636.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sample grid – 5 random images per class


In [18]:
def show_grid(cls_dict, title):
    classes = list(cls_dict.keys())
    n = len(classes)
    fig, axs = plt.subplots(n, 5, figsize=(12, 2.2 * n))
    if n == 1:
        axs = [axs]
    for row, cls in enumerate(classes):
        samples = random.sample(cls_dict[cls], min(5, len(cls_dict[cls])))
        for col in range(5):
            ax = axs[row][col]
            if col < len(samples):
                try:
                    img = Image.open(samples[col]).convert('RGB')
                    ax.imshow(img)
                except Exception:
                    ax.text(0.5, 0.5, 'err', ha='center')
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(cls, rotation=0, labelpad=60, fontsize=8, va='center')
                ax.yaxis.set_label_position('left')
                ax.set_yticks([])
    plt.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{title.lower().replace(" ","_")}_grid.png', dpi=80, bbox_inches='tight')
    plt.show()

show_grid(cucumber_cls, 'Cucumber Samples')
show_grid(sunflower_cls, 'Sunflower Samples')


C:\Users\akank\AppData\Local\Temp\ipykernel_19272\2064869287.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Unique image dimensions


In [19]:
def unique_dims(cls_dict, sample_limit=100):
    dims = set()
    for paths in cls_dict.values():
        for p in paths[:sample_limit]:
            try:
                with Image.open(p) as im:
                    dims.add(im.size)
            except Exception:
                pass
    return dims

c_dims = unique_dims(cucumber_cls)
s_dims = unique_dims(sunflower_cls)
print(f'Cucumber – {len(c_dims)} unique dimension(s):', sorted(c_dims))
print(f'Sunflower – {len(s_dims)} unique dimension(s):', sorted(s_dims))


Cucumber – 116 unique dimension(s): [(960, 720), (1267, 1712), (1611, 2298), (1658, 2206), (1677, 1733), (1689, 1590), (1707, 2072), (1711, 2143), (1713, 2792), (1722, 2383), (1741, 2063), (1749, 1753), (1749, 2018), (1751, 2300), (1764, 1997), (1765, 2316), (1789, 2536), (1801, 1762), (1808, 1768), (1809, 1466), (1811, 2381), (1819, 2108), (1822, 2191), (1825, 2129), (1825, 2277), (1827, 2466), (1830, 2236), (1840, 2505), (1854, 2108), (1857, 2501), (1859, 2229), (1863, 2450), (1866, 2348), (1878, 2243), (1883, 2446), (1887, 2915), (1895, 2106), (1897, 1802), (1899, 1676), (1910, 2002), (1917, 2944), (1936, 1875), (1939, 2176), (1944, 1945), (1944, 2359), (1948, 2238), (1949, 1836), (1949, 2071), (1950, 2256), (1951, 1682), (1951, 2185), (1951, 2377), (1951, 2529), (1952, 2240), (1956, 2128), (1956, 2427), (1956, 2846), (1958, 2879), (2044, 1706), (2111, 1721), (2141, 1700), (2187, 1789), (2190, 2788), (2217, 1455), (2221, 2798), (2324, 3716), (2534, 3459), (2561, 1317), (2577, 4029),

## Mean image per class


In [20]:
TARGET_SIZE = (128, 128)

def mean_img_per_class(cls_dict, limit=50):
    means = {}
    for cls, paths in cls_dict.items():
        arrs = []
        for p in paths[:limit]:
            try:
                with Image.open(p) as im:
                    arrs.append(np.array(im.convert('RGB').resize(TARGET_SIZE), dtype=np.float32))
            except Exception:
                pass
        if arrs:
            means[cls] = np.mean(arrs, axis=0).astype(np.uint8)
    return means

c_means = mean_img_per_class(cucumber_cls)
s_means = mean_img_per_class(sunflower_cls)

def plot_means(means, title):
    n = len(means)
    fig, axs = plt.subplots(1, n, figsize=(3*n, 3))
    if n == 1: axs = [axs]
    for ax, (cls, arr) in zip(axs, means.items()):
        ax.imshow(arr)
        ax.set_title(cls, fontsize=8)
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(f'{title.lower().replace(" ","_")}.png', dpi=80, bbox_inches='tight')
    plt.show()

plot_means(c_means, 'Cucumber Mean Images')
plot_means(s_means, 'Sunflower Mean Images')


C:\Users\akank\AppData\Local\Temp\ipykernel_19272\1309127842.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## RGB channel histograms per class


In [21]:
def plot_rgb_hist(cls_dict, title, limit=30):
    classes = list(cls_dict.keys())
    n = len(classes)
    fig, axs = plt.subplots(n, 3, figsize=(12, 2.5*n), sharex=True)
    if n == 1: axs = [axs]
    ch_names = ['Red', 'Green', 'Blue']
    ch_colors = ['red', 'green', 'blue']
    for row, cls in enumerate(classes):
        channels = [[], [], []]
        for p in cls_dict[cls][:limit]:
            try:
                with Image.open(p) as im:
                    arr = np.array(im.convert('RGB'))
                    for c in range(3):
                        channels[c].append(arr[:,:,c].ravel())
            except Exception:
                pass
        for col in range(3):
            ax = axs[row][col]
            if channels[col]:
                ax.hist(np.concatenate(channels[col]), bins=64, color=ch_colors[col], alpha=0.7)
            ax.set_title(f'{cls} – {ch_names[col]}', fontsize=8)
            ax.set_xlim(0, 255)
    plt.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{title.lower().replace(" ","_")}_hist.png', dpi=80, bbox_inches='tight')
    plt.show()

plot_rgb_hist(cucumber_cls, 'Cucumber')
plot_rgb_hist(sunflower_cls, 'Sunflower')


C:\Users\akank\AppData\Local\Temp\ipykernel_19272\1846703231.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\akank\AppData\Local\Temp\ipykernel_19272\1846703231.py:4: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axs = plt.subplots(n, 3, figsize=(12, 2.5*n), sharex=True)


## Mean brightness and contrast per class


In [22]:
def brightness_contrast(cls_dict, limit=50):
    rows = []
    for cls, paths in cls_dict.items():
        br, co = [], []
        for p in paths[:limit]:
            try:
                with Image.open(p) as im:
                    st = ImageStat.Stat(im.convert('L'))
                    br.append(st.mean[0])
                    co.append(st.stddev[0])
            except Exception:
                pass
        rows.append({'Class': cls, 'Mean Brightness': round(np.mean(br), 2) if br else None,
                     'Mean Contrast (StdDev)': round(np.mean(co), 2) if co else None})
    return pd.DataFrame(rows).set_index('Class')

print('=== Cucumber ===' )
c_stats = brightness_contrast(cucumber_cls)
print(c_stats.to_string())

print('\n=== Sunflower ===')
s_stats = brightness_contrast(sunflower_cls)
print(s_stats.to_string())


=== Cucumber ===
                  Mean Brightness  Mean Contrast (StdDev)
Class                                                    
Flowering                  162.58                   54.12
Fruiting - 1               141.67                   62.69
Fruiting - 2               137.18                   67.43
Healthy Leaves             148.08                   58.56
Unhealthy Leaves           144.08                   58.02

=== Sunflower ===
            Mean Brightness  Mean Contrast (StdDev)
Class                                              
EarlyBloom           104.30                   50.29
Healthy              104.37                   53.28
MatureBud            113.44                   45.11
Wilted               106.09                   50.03
YoungBud             109.67                   44.66
